In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile /content/drive/MyDrive/mri/oasis_freesurfer_urls.txt
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc1.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc2.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc3.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc4.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc5.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc6.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc7.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc8.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc9.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc10.tar.gz
https://download.nrg.wustl.edu/data/oasis_cs_freesurfer_disc11.tar.gz

Writing /content/drive/MyDrive/mri/oasis_freesurfer_urls.txt


In [ ]:
%%bash
DEST_DIR="/content/drive/MyDrive/mri_data/oasis_freesurfer"
URL_FILE="/content/drive/MyDrive/mri/oasis_freesurfer_urls.txt"

if [ ! -f "$URL_FILE" ]; then
    echo "[-] Ошибка: Файл '$URL_FILE' не найденd."
    exit 1
fi

echo "[+] Начало загрузки..."
wget \
    --input-file="$URL_FILE" \
    --directory-prefix="$DEST_DIR"

echo "[+] Загрузка закончена. Файлы на Google Drive."

[+] Начало загрузки...
[+] Загрузка закончена. Файлы на Google Drive.


In [ ]:
# Проверим загруженные архивы
%%bash
ls -lh "/content/drive/MyDrive/mri_data/oasis_freesurfer"

total 102G
-rw------- 1 root root 8.5G Jul 20  2007 oasis_cs_freesurfer_disc10.tar.gz
-rw------- 1 root root 9.3G Jul 20  2007 oasis_cs_freesurfer_disc11.tar.gz
-rw------- 1 root root 9.4G Jul 20  2007 oasis_cs_freesurfer_disc1.tar.gz
-rw------- 1 root root 9.1G Jul 20  2007 oasis_cs_freesurfer_disc2.tar.gz
-rw------- 1 root root 9.3G Jul 20  2007 oasis_cs_freesurfer_disc3.tar.gz
-rw------- 1 root root 9.6G Jul 20  2007 oasis_cs_freesurfer_disc4.tar.gz
-rw------- 1 root root 9.2G Jul 21  2007 oasis_cs_freesurfer_disc5.tar.gz
-rw------- 1 root root 9.6G Jul 21  2007 oasis_cs_freesurfer_disc6.tar.gz
-rw------- 1 root root 9.6G Jul 21  2007 oasis_cs_freesurfer_disc7.tar.gz
-rw------- 1 root root 9.0G Jul 21  2007 oasis_cs_freesurfer_disc8.tar.gz
-rw------- 1 root root 8.8G Jul 21  2007 oasis_cs_freesurfer_disc9.tar.gz


In [ ]:
# Распакуем архивы
%%bash
SOURCE_DIR="/content/drive/MyDrive/mri_data/oasis_freesurfer"
DEST_DIR="/content/drive/MyDrive/mri/oasis_freesurfer"

if [ ! -d "$SOURCE_DIR" ]; then
    echo "Ошибка: Директория не существует: $SOURCE_DIR"
    exit 1
fi

files=("$SOURCE_DIR"/*.tar.gz)

if [ ${#files[@]} -eq 0 ]; then
    echo "Файл .tar.gz не найден в $SOURCE_DIR"
    exit 0
fi

for archive in "${files[@]}"; do
    echo "----------------------------------------"
    echo "Распаковываем архив: $(basename "$archive")"

    tar -xzf "$archive" -C "$DEST_DIR"

    if [ $? -eq 0 ]; then
        echo "Распакован успешно: $(basename "$archive")"
    else
        echo "Ошибка при распаковки: $(basename "$archive")"
    fi
done

echo "----------------------------------------"
echo "Распаковка всех архивов закончена."

----------------------------------------
Распаковываем архив: oasis_cs_freesurfer_disc10.tar.gz
Распакован успешно: oasis_cs_freesurfer_disc10.tar.gz
----------------------------------------
Распаковываем архив: oasis_cs_freesurfer_disc11.tar.gz
Распакован успешно: oasis_cs_freesurfer_disc11.tar.gz
----------------------------------------
Распаковываем архив: oasis_cs_freesurfer_disc1.tar.gz
Распакован успешно: oasis_cs_freesurfer_disc1.tar.gz
----------------------------------------
Распаковываем архив: oasis_cs_freesurfer_disc2.tar.gz
Распакован успешно: oasis_cs_freesurfer_disc2.tar.gz
----------------------------------------
Распаковываем архив: oasis_cs_freesurfer_disc3.tar.gz
Распакован успешно: oasis_cs_freesurfer_disc3.tar.gz
----------------------------------------
Распаковываем архив: oasis_cs_freesurfer_disc4.tar.gz
Распакован успешно: oasis_cs_freesurfer_disc4.tar.gz
----------------------------------------
Распаковываем архив: oasis_cs_freesurfer_disc5.tar.gz
Распакован ус

In [4]:
# проверяем что все архивы с дисками распакованы
!ls "/content/drive/MyDrive/mri/oasis_freesurfer"

disc1  disc10  disc11  disc2  disc3  disc4  disc5  disc6  disc7  disc8	disc9


In [ ]:
# Для упрощения дальнейшей работы скопируем все подпапки с данными субьектов в одну папку
%%bash
cd /content/drive/MyDrive/mri/oasis_freesurfer
echo "Текущая директория:"
pwd
echo "[+] Начало копирования..."
SOURCES=(disc1 disc2 disc3 disc4 disc5 disc6 disc7 disc8 disc9 disc10 disc11)
DESTINATION="/content/drive/My Drive/mri/processed_oasis_data"

for src in "${SOURCES[@]}"; do
    if [ -d "$src" ]; then
        echo "[+] Копируем данные из папки: $src"

        cp -a "$src/." "$DESTINATION/"
    else
        echo "Ошибка: Папка с диском $src не найдена, пропускаем"
    fi
done

echo "[+] Копирование завершено"


Текущая директория:
/content/drive/MyDrive/mri/oasis_freesurfer
[+] Начало копирования...
[+] Копируем данные из папки: disc1
[+] Копируем данные из папки: disc2
[+] Копируем данные из папки: disc3
[+] Копируем данные из папки: disc4
[+] Копируем данные из папки: disc5
[+] Копируем данные из папки: disc6
[+] Копируем данные из папки: disc7
[+] Копируем данные из папки: disc8
[+] Копируем данные из папки: disc9
[+] Копируем данные из папки: disc10
[+] Копируем данные из папки: disc11
[+] Копирование завершено


In [ ]:
# Проверим что все папки с данными субьектов в одной директории
!ls -h "/content/drive/MyDrive/mri/oasis_processed_all_subjects"

OAS1_0001_MR1  OAS1_0090_MR1  OAS1_0178_MR1  OAS1_0270_MR1  OAS1_0363_MR1
OAS1_0002_MR1  OAS1_0091_MR1  OAS1_0179_MR1  OAS1_0271_MR1  OAS1_0365_MR1
OAS1_0003_MR1  OAS1_0092_MR1  OAS1_0180_MR1  OAS1_0272_MR1  OAS1_0366_MR1
OAS1_0004_MR1  OAS1_0092_MR2  OAS1_0181_MR1  OAS1_0273_MR1  OAS1_0367_MR1
OAS1_0005_MR1  OAS1_0094_MR1  OAS1_0182_MR1  OAS1_0274_MR1  OAS1_0368_MR2
OAS1_0006_MR1  OAS1_0095_MR1  OAS1_0183_MR1  OAS1_0275_MR1  OAS1_0369_MR1
OAS1_0007_MR1  OAS1_0096_MR1  OAS1_0184_MR1  OAS1_0277_MR1  OAS1_0370_MR1
OAS1_0009_MR1  OAS1_0097_MR1  OAS1_0186_MR1  OAS1_0278_MR1  OAS1_0371_MR1
OAS1_0010_MR1  OAS1_0098_MR1  OAS1_0188_MR1  OAS1_0279_MR1  OAS1_0372_MR1
OAS1_0011_MR1  OAS1_0099_MR1  OAS1_0189_MR1  OAS1_0280_MR1  OAS1_0374_MR1
OAS1_0012_MR1  OAS1_0101_MR1  OAS1_0190_MR1  OAS1_0281_MR1  OAS1_0375_MR1
OAS1_0013_MR1  OAS1_0101_MR2  OAS1_0191_MR1  OAS1_0282_MR1  OAS1_0376_MR1
OAS1_0014_MR1  OAS1_0102_MR1  OAS1_0191_MR2  OAS1_0283_MR1  OAS1_0377_MR1
OAS1_0015_MR1  OAS1_0103_MR1  OAS1_019

In [ ]:
# Загрузка файла с демографией субьектов и клиническими данныим (Demographic and Clinical Data)
!wget "https://sites.wustl.edu/oasisbrains/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx" \
  --directory-prefix="/content/drive/MyDrive/mri"

--2026-11-09 08:18:54--  https://sites.wustl.edu/oasisbrains/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx
Resolving sites.wustl.edu (sites.wustl.edu)... 3.141.160.164, 3.149.144.83
Connecting to sites.wustl.edu (sites.wustl.edu)|3.141.160.164|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://bpb-us-e2.wpmucdn.com/sites.wustl.edu/dist/6/4383/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx [following]
--2026-11-09 08:18:55--  https://bpb-us-e2.wpmucdn.com/sites.wustl.edu/dist/6/4383/files/2024/04/oasis_cross-sectional-5708aa0a98d82080.xlsx
Resolving bpb-us-e2.wpmucdn.com (bpb-us-e2.wpmucdn.com)... 38.104.169.186, 2001:550:2:6f::f3:2
Connecting to bpb-us-e2.wpmucdn.com (bpb-us-e2.wpmucdn.com)|38.104.169.186|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 35716 (35K) [application/vnd.openxmlformats-officedocument.spreadsheetml.sheet]
Saving to: ‘/content/drive/MyDrive/mri/oasis_cross-sectional-5708aa0a98d

In [ ]:
# Загрузка файла с повторным сканированием и поправками (Reliability Data Spreadsheet)
!wget "https://sites.wustl.edu/oasisbrains/files/2024/04/oasis_cross-sectional-reliability-063c8642b909ee76.xlsx" \
  --directory-prefix="/content/drive/MyDrive/mri"

--2026-11-09 08:20:25--  https://sites.wustl.edu/oasisbrains/files/2024/04/oasis_cross-sectional-reliability-063c8642b909ee76.xlsx
Resolving sites.wustl.edu (sites.wustl.edu)... 3.141.160.164, 3.149.144.83
Connecting to sites.wustl.edu (sites.wustl.edu)|3.141.160.164|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://bpb-us-e2.wpmucdn.com/sites.wustl.edu/dist/6/4383/files/2024/04/oasis_cross-sectional-reliability-063c8642b909ee76.xlsx [following]
--2026-11-09 08:20:25--  https://bpb-us-e2.wpmucdn.com/sites.wustl.edu/dist/6/4383/files/2024/04/oasis_cross-sectional-reliability-063c8642b909ee76.xlsx
Resolving bpb-us-e2.wpmucdn.com (bpb-us-e2.wpmucdn.com)... 38.104.169.186, 2001:550:2:6f::f3:2
Connecting to bpb-us-e2.wpmucdn.com (bpb-us-e2.wpmucdn.com)|38.104.169.186|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10680 (10K) [application/vnd.openxmlformats-officedocument.spreadsheetml.sheet]
Saving to: ‘/content/drive/MyDrive/m

In [ ]:
# Загрузка файла с описанием файлов OASIS-1
!wget "https://sites.wustl.edu/oasisbrains/files/2024/03/oasis_cross-sectional_facts-bcc7a002dfb104f4.pdf" \
  --directory-prefix="/content/drive/MyDrive/mri"

--2026-11-09 08:30:07--  https://sites.wustl.edu/oasisbrains/files/2024/03/oasis_cross-sectional_facts-bcc7a002dfb104f4.pdf
Resolving sites.wustl.edu (sites.wustl.edu)... 3.141.160.164, 3.149.144.83
Connecting to sites.wustl.edu (sites.wustl.edu)|3.141.160.164|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://bpb-us-e2.wpmucdn.com/sites.wustl.edu/dist/6/4383/files/2024/03/oasis_cross-sectional_facts-bcc7a002dfb104f4.pdf [following]
--2026-11-09 08:30:07--  https://bpb-us-e2.wpmucdn.com/sites.wustl.edu/dist/6/4383/files/2024/03/oasis_cross-sectional_facts-bcc7a002dfb104f4.pdf
Resolving bpb-us-e2.wpmucdn.com (bpb-us-e2.wpmucdn.com)... 38.104.169.186, 2001:550:2:6f::f3:2
Connecting to bpb-us-e2.wpmucdn.com (bpb-us-e2.wpmucdn.com)|38.104.169.186|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 99660 (97K) [application/pdf]
Saving to: ‘/content/drive/MyDrive/mri/oasis_cross-sectional_facts-bcc7a002dfb104f4.pdf’

oasis_cross-secti